# 4. Which depth's PV-gradient direction best matches tilt?

`TiltDir` is the existing whole-column tilt direction. Agreement is polarity-aware: AEs are expected along signed grad(PV), CEs opposite it. Each eddy contributes equally to the summary.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import seacofs_tilt_tools as tilt
import depth_pv_tools as dpt

sns.set_theme(style="whitegrid", context="notebook")
DOMINANCE_FACTOR = 2.0
TARGET_DEPTHS_M = (0, 200, 500, 700, 1000)
MIN_TILT_KM = 5.0

depth_df = tilt.add_pv_gradient_terms(source="depth")
snapshot_df = tilt.add_pv_gradient_terms(source="depth_snapshot")
dpt.validate_depth_tables(depth_df, snapshot_df)
DEPTHS = dpt.nearest_cached_depths(depth_df, TARGET_DEPTHS_M)
comparison = dpt.add_surface_differences(depth_df, DOMINANCE_FACTOR)
matched = dpt.matched_depth_rows(comparison, DEPTHS)
palette = {"AE": "#c44e52", "CE": "#4c72b0"}
depth_cmap = plt.get_cmap("viridis")
depth_colours = dict(zip(DEPTHS, depth_cmap(np.linspace(.08, .92, len(DEPTHS)))))
depth_labels = {z: f"{z:g} m" for z in DEPTHS}


In [ ]:
direction = matched[matched.TiltDis.ge(MIN_TILT_KM)].copy()
score = dpt.directional_scorecard(direction)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for cyc, part in score.groupby("Cyc"):
    axes[0].plot(part.Depth, part.median_error_deg, marker="o", lw=2, color=palette[cyc], label=cyc)
    axes[0].fill_between(part.Depth, part.q25_error_deg, part.q75_error_deg, color=palette[cyc], alpha=.18)
    axes[1].plot(part.Depth, part.fraction_within_45, marker="o", lw=2, color=palette[cyc], label=cyc)
axes[0].set(xlabel="PV sampling depth (m)", ylabel="Median expected-response error (degrees)", title="Lower is better")
axes[1].set(xlabel="PV sampling depth (m)", ylabel="Eddy fraction within 45°", title="Higher is better", ylim=(0,1))
axes[0].legend(frameon=False); fig.suptitle("Polarity-aware directional agreement with measured tilt")
plt.show()

In [ ]:
rose_depths = [DEPTHS[0], DEPTHS[-1]]
fig, axes = plt.subplots(2, 2, figsize=(9, 8), subplot_kw={"projection":"polar"}, constrained_layout=True)
bins = np.linspace(-180, 180, 25)
for row, cyc in enumerate(["AE", "CE"]):
    for col, z in enumerate(rose_depths):
        part = direction[(direction.Cyc == cyc) & np.isclose(direction.Depth, z)]
        eddy_angle = part.groupby("Eddy").preference_error_deg.median()
        signed = np.r_[eddy_angle, -eddy_angle]
        counts, edges = np.histogram(np.deg2rad(signed), bins=np.deg2rad(bins), density=True)
        axes[row,col].bar((edges[:-1]+edges[1:])/2, counts, width=np.diff(edges), color=palette[cyc], alpha=.72)
        axes[row,col].set_theta_zero_location("N"); axes[row,col].set_theta_direction(-1)
        axes[row,col].set_title(f"{cyc}, {z:g} m")
fig.suptitle("Absolute expected-response error mirrored for visual comparison")
plt.show()

A depth-dependent minimum in directional error would identify a better explanatory sampling level. Interpret effect size and consistency across polarities, not only significance.